# 29B. Text-Table In-Batch Mining — Remaining Sizes

Adapted from **NB 29** (text-image in-batch mining) for **text-table** pairs.

### Differences from NB 29

| Axis | NB 29 | NB 29B |
|---|---|---|
| Pair type | text ↔ image | **text ↔ table** |
| Encoders | DINOv3 + BGE-M3 | **BGE-M3 only** |
| Heads | `head_text_bge`, `head_image_dinov3` | **`head_text_bge_v2`, `head_table_bge_v2`** |
| Season test | `season_i == season_j` | **`delta_month(month_i, month_j) < 2`** (circular) |
| GPU | `cuda:1` | **`cuda:0`** (matches NB 28) |
| Log file | `<OUTPUT_ROOT>/img_inbatch_mining_ft.log` | **`<OUTPUT_DIR>/texttable_inbatch_mining_ft.log`** (inside the sweep dir) |
| BGE max-len | 512 | **1024** (tables are longer) |

### What is unchanged from NB 29

- In-batch mined contrastive loss (symmetric InfoNCE with hard/soft/drop classification)
- Per-batch hard/soft counting accumulated into `epoch_stats` / `history`
- Skip-if-`gt_results.csv`-exists, resume-from-checkpoint, atomic checkpoint writes
- Per-size artefacts layout: `size_{N}/{latest_checkpoint.pt, training_history.csv, gt_results.csv, *.pt}`
- Leak filtering via `leak_df["row_index"]` vs `orig_idx`

### Prerequisites

| Pickle | Producer |
|---|---|
| `vars/bulletin_to_meta_raw.pkl` | NB 29 Section 2 |
| `vars/texttable_records.pkl` | NB 28 Section 2 |
| `vars/gt_texttable_records.pkl` | NB 28 Section 6 |

In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


## Section 0 — Installations

In [ ]:
# !pip install openpyxl
# !pip install pandas --upgrade
# !pip install accelerate
# !pip install transformers==4.57.6
# !pip install --upgrade huggingface_hub

## Section 1 — Setup, logging, config

Log file lives inside `texttable_inbatch_mining_sweep/`. GPU pinned to `cuda:0`.

In [2]:
import os
import re
import sys
import time
import math
import random
import pickle
import logging
import gc
from collections import defaultdict
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
import transformers

In [3]:
transformers.logging.set_verbosity_error()

# --- Paths: define BOTH root and sweep dir up front so the log can live inside the sweep ---
OUTPUT_ROOT = str(RESULTS_RUNS_ROOT)
OUTPUT_DIR  = os.path.join(OUTPUT_ROOT, "mined_negatives_ft", "texttable_inbatch_mining_sweep")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)

LOG_FORMAT = '%(asctime)s : %(levelname)s : %(message)s'
LOG_FILE   = os.path.join(OUTPUT_DIR, "texttable_inbatch_mining_ft.log")

logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format=LOG_FORMAT)
logger = logging.getLogger("texttable_inbatch_ft")
if not any(type(h) is logging.StreamHandler for h in logger.handlers):
    _sh = logging.StreamHandler(sys.stdout)
    _sh.setFormatter(logging.Formatter(LOG_FORMAT))
    logger.addHandler(_sh)


# --- GPU pinning: always cuda:0 (matches NB 28) ---
CUDA = "cuda:0"
device = CUDA if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)
logger.info("Device: %s", device)
if device == CUDA:
    logger.info("GPU: %s (physical index %d of %d visible)",
                torch.cuda.get_device_name(torch.cuda.current_device()),
                torch.cuda.current_device(),
                torch.cuda.device_count())

logger.info("LOG_FILE = %s", LOG_FILE)
logger.info("OUTPUT_DIR = %s", OUTPUT_DIR)


def pkl_vars(varname, filename):
    with open(filename, 'wb') as file:
        pickle.dump(varname, file)

def reload_vars(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

2026-10-23 18:43:33,683 : INFO : Device: cuda:0
2026-10-23 18:43:33,687 : INFO : GPU: Quadro RTX 8000 (physical index 0 of 2 visible)
2026-10-23 18:43:33,688 : INFO : LOG_FILE = /projects/agrifutur/sji/cross_encoder_ft/mined_negatives_ft/texttable_inbatch_mining_sweep/texttable_inbatch_mining_ft.log
2026-10-23 18:43:33,689 : INFO : OUTPUT_DIR = /projects/agrifutur/sji/cross_encoder_ft/mined_negatives_ft/texttable_inbatch_mining_sweep


### Config

In [4]:
# --- Shared ---
HF_CACHE          = DEFAULT_HF_CACHE
RANDOM_SEED       = 42
MAX_RUNTIME_HOURS_PER_SIZE = 7.5

BGE_MODEL_NAME    = "BAAI/bge-m3"
BGE_MAX_LENGTH    = 1024
N_UNFROZEN_LAYERS_BGE = 6

SHARED_DIM        = 512
BATCH_SIZE        = 32
HEAD_LEARNING_RATE = 1e-4
BGE_ENCODER_LR    = 1e-5
TEMPERATURE       = 0.07

K_VALUES = [1, 3, 5, 10]

LOG_EVERY_N_STEPS        = 500
CHECKPOINT_EVERY_N_STEPS = 500

_MODEL_FILE_RENAMES = {
    "bge_model": "bge_model_unfrozen",
}

# --- INPUT paths ---
VARS_DIR = str(FINETUNING_ROOT / "vars")
os.makedirs(VARS_DIR, exist_ok=True)

LEAK_FLAGS_PATH = str(FINETUNING_ROOT / 'Shared_data' / 'table_leak_flags_COMPLETE.xlsx')

SIZES_TO_SWEEP  = [100, 500, 1000, 2500, 5000, 6374]  

logger.info("Config loaded. OUTPUT_DIR=%s", OUTPUT_DIR)
logger.info("BATCH_SIZE=%d, BGE_MAX_LENGTH=%d, N_UNFROZEN_LAYERS_BGE=%d",
            BATCH_SIZE, BGE_MAX_LENGTH, N_UNFROZEN_LAYERS_BGE)
logger.info("SIZES_TO_SWEEP=%s", SIZES_TO_SWEEP)
logger.info("LEAK_FLAGS_PATH exists: %s", os.path.exists(LEAK_FLAGS_PATH))

2026-10-23 18:43:42,692 : INFO : Config loaded. OUTPUT_DIR=/projects/agrifutur/sji/cross_encoder_ft/mined_negatives_ft/texttable_inbatch_mining_sweep
2026-10-23 18:43:42,694 : INFO : BATCH_SIZE=32, BGE_MAX_LENGTH=1024, N_UNFROZEN_LAYERS_BGE=6
2026-10-23 18:43:42,696 : INFO : SIZES_TO_SWEEP=[100, 500, 1000, 2500, 5000, 6374]
2026-10-23 18:43:42,698 : INFO : LEAK_FLAGS_PATH exists: True


## Section 2 — Load bulletin -> month/region mapping

Reuses `vars/bulletin_to_meta_raw.pkl` produced by NB 29's parse step. Derives **month** from `date`.

In [5]:
BULLETIN_META_RAW_PATH = os.path.join(VARS_DIR, "bulletin_to_meta_raw.pkl")
if not os.path.exists(BULLETIN_META_RAW_PATH):
    raise FileNotFoundError(
        f"Missing {BULLETIN_META_RAW_PATH}.\n"
        "Run NB 29's Section 2 once to build it, then re-run this cell."
    )

bulletin_to_meta_raw = reload_vars(BULLETIN_META_RAW_PATH)
logger.info("Loaded %d bulletin -> {date, region_qid} mappings", len(bulletin_to_meta_raw))

bulletin_meta_ext = {}
for bid, meta in bulletin_to_meta_raw.items():
    try:
        dt = datetime.strptime(meta["date"], "%Y-%m-%d")
    except (ValueError, KeyError):
        continue
    bulletin_meta_ext[bid] = {"month": dt.month, "region_qid": meta["region_qid"]}

logger.info("Enriched with month: %d bulletins", len(bulletin_meta_ext))

2026-10-23 18:43:47,437 : INFO : Loaded 20320 bulletin -> {date, region_qid} mappings
2026-10-23 18:43:47,539 : INFO : Enriched with month: 20320 bulletins


## Section 3 — Load text-table data, enrich, leak-filter

Loads NB 28's `texttable_records.pkl` (6757 records), enriches with `month`/`region_qid`, then applies the same leak filter NB 29 uses.

In [6]:
TEXTTABLE_RECORDS_PATH = os.path.join(VARS_DIR, "texttable_records.pkl")
if not os.path.exists(TEXTTABLE_RECORDS_PATH):
    raise FileNotFoundError(
        f"Missing {TEXTTABLE_RECORDS_PATH}.\n"
        "Run NB 28's Section 2 once to build it, then re-run this cell."
    )

raw_texttable_records = reload_vars(TEXTTABLE_RECORDS_PATH)
logger.info("Loaded %d texttable_records from NB 28 pickle", len(raw_texttable_records))

# --- Enrich with month + region_qid ---
texttable_records = []
n_missing_meta = 0
for r in raw_texttable_records:
    bid = r.get("bulletin_id")
    if bid not in bulletin_meta_ext:
        n_missing_meta += 1
        continue
    texttable_records.append({
        **r,
        "month":      bulletin_meta_ext[bid]["month"],
        "region_qid": bulletin_meta_ext[bid]["region_qid"],
    })

logger.info("Enriched texttable_records: %d (%d dropped: bulletin_id not in meta)",
            len(texttable_records), n_missing_meta)

# --- Leak filter (matches NB 29's approach) ---
leak_df = pd.read_excel(LEAK_FLAGS_PATH)
leaked_indices = set(leak_df["row_index"])

before_leak_count = len(texttable_records)
texttable_records = [r for r in texttable_records if r["orig_idx"] not in leaked_indices]
after_leak_count = len(texttable_records)

logger.info("Dropped %d leaked rows (%d flagged by row_index)",
            before_leak_count - after_leak_count, len(leaked_indices))
logger.info("Text/table records: %d -> %d (after leak filter)",
            before_leak_count, after_leak_count)

if before_leak_count - after_leak_count == 0:
    logger.warning("Leak filter dropped 0 rows. If leak flags were computed against the textimage "
                   "jsonl, the row_index values may not match texttable orig_idx.")

# --- Shuffle + record pool size ---
random.seed(RANDOM_SEED)
random.shuffle(texttable_records)

FULL_POOL_SIZE = len(texttable_records)
logger.info("Full texttable pool (post-leak-filter, shuffled): %d records", FULL_POOL_SIZE)

_sample = texttable_records[0]
for _k in ("bulletin_id", "text", "table_text", "month", "region_qid"):
    assert _k in _sample, f"missing key {_k!r} in texttable_records"
logger.info("Training record schema OK")

_oversized = [N for N in SIZES_TO_SWEEP if N >= FULL_POOL_SIZE]
if _oversized:
    logger.warning("Sweep sizes >= FULL_POOL_SIZE (%d) will use the full pool: %s",
                   FULL_POOL_SIZE, _oversized)

2026-10-23 18:43:48,365 : INFO : Loaded 6757 texttable_records from NB 28 pickle
2026-10-23 18:43:48,375 : INFO : Enriched texttable_records: 6694 (63 dropped: bulletin_id not in meta)
2026-10-23 18:43:49,800 : INFO : Dropped 320 leaked rows (922 flagged by row_index)
2026-10-23 18:43:49,802 : INFO : Text/table records: 6694 -> 6374 (after leak filter)
2026-10-23 18:43:49,806 : INFO : Full texttable pool (post-leak-filter, shuffled): 6374 records
2026-10-23 18:43:49,807 : INFO : Training record schema OK
2026-10-23 18:43:49,809 : WARNING : Sweep sizes >= FULL_POOL_SIZE (6374) will use the full pool: [6374]


## Section 4 — In-batch negative classification (month-distance season test)

**Change vs. NB 29**: `same_season` is now `delta_month(m1, m2) < 2` (circular), where `different_season` is `delta >= 2`.

- **drop**: `same_season AND same_region`
- **hard**: `same_season XOR same_region`
- **soft**: `different_season AND different_region`

In [7]:
def _month_delta(m1, m2):
    """Circular distance between two months (1-12), range [0, 6]."""
    d = abs(m1 - m2)
    return min(d, 12 - d)


def classify_batch_negatives(batch):
    """Returns {anchor_batch_idx: {"hard": [...], "soft": [...]}}."""
    B = len(batch)
    result = {}
    for i in range(B):
        hard, soft = [], []
        for j in range(B):
            if i == j:
                continue
            delta_m = _month_delta(batch[i]["month"], batch[j]["month"])
            same_season = (delta_m < 2)
            same_region = batch[i]["region_qid"] == batch[j]["region_qid"]

            if same_season and same_region:
                continue
            if same_season != same_region:
                hard.append(j)
            else:
                soft.append(j)
        result[i] = {"hard": hard, "soft": soft}
    return result

logger.info("classify_batch_negatives() ready (month-distance season test, threshold=2).")

2026-10-23 18:43:49,815 : INFO : classify_batch_negatives() ready (month-distance season test, threshold=2).


## Section 5 — Model builder (BGE-M3 only, 2 heads)

Same architecture as NB 28's texttable branch. Unfreezes the last 6 BGE encoder layers.

In [8]:
def build_models():
    bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE)
    bge_model = AutoModel.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE).to(device)

    for p in bge_model.parameters():
        p.requires_grad = False
    for layer in bge_model.encoder.layer[-N_UNFROZEN_LAYERS_BGE:]:
        for p in layer.parameters():
            p.requires_grad = True

    BGE_DIM = bge_model.config.hidden_size

    def encode_bge_text(texts):
        inputs = bge_tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                                 max_length=BGE_MAX_LENGTH).to(device)
        out = bge_model(**inputs)
        return out.last_hidden_state[:, 0, :]

    class ProjectionHead(nn.Module):
        def __init__(self, in_dim, out_dim=SHARED_DIM):
            super().__init__()
            self.proj = nn.Linear(in_dim, out_dim)
        def forward(self, x):
            return F.normalize(self.proj(x), p=2, dim=-1)

    head_text_bge_v2  = ProjectionHead(BGE_DIM).to(device)
    head_table_bge_v2 = ProjectionHead(BGE_DIM).to(device)

    unfrozen_bge_params = [p for p in bge_model.parameters() if p.requires_grad]
    head_params = list(head_text_bge_v2.parameters()) + list(head_table_bge_v2.parameters())

    optimizer = torch.optim.AdamW([
        {"params": head_params, "lr": HEAD_LEARNING_RATE},
        {"params": unfrozen_bge_params, "lr": BGE_ENCODER_LR},
    ])

    logger.info("build_models: BGE trainable params=%d, optimizer groups=2", len(unfrozen_bge_params))

    return {
        "bge_model": bge_model, "encode_bge_text": encode_bge_text,
        "head_text_bge_v2": head_text_bge_v2, "head_table_bge_v2": head_table_bge_v2,
        "optimizer": optimizer,
    }

logger.info("build_models() ready.")

2026-10-23 18:43:49,845 : INFO : build_models() ready.


## Section 6 — Batch construction + in-batch mined loss

In [9]:
def make_batches(records, batch_size=BATCH_SIZE, shuffle=True):
    idx = np.arange(len(records))
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        batch_idx = idx[start:start + batch_size]
        yield [records[i] for i in batch_idx]


def compute_texttable_loss_inbatch_mined(batch, models, batch_stats):
    """Text-table in-batch mined loss. Accumulates hard/soft/drop counts into batch_stats."""
    B = len(batch)
    texts  = [r["text"] for r in batch]
    tables = [r["table_text"] for r in batch]

    text_emb_raw  = models["encode_bge_text"](texts)
    table_emb_raw = models["encode_bge_text"](tables)

    text_emb  = models["head_text_bge_v2"](text_emb_raw)
    table_emb = models["head_table_bge_v2"](table_emb_raw)

    sim = text_emb @ table_emb.T

    classification = classify_batch_negatives(batch)

    mask = torch.zeros((B, B), dtype=torch.bool, device=device)
    for i in range(B):
        info = classification[i]
        hard_n, soft_n = len(info["hard"]), len(info["soft"])
        batch_stats["hard_count"] += hard_n
        batch_stats["soft_count"] += soft_n
        batch_stats["dropped_count"] += (B - 1 - hard_n - soft_n)
        if hard_n == 0:
            batch_stats["zero_hard_anchors"] += 1
        if hard_n + soft_n == 0:
            batch_stats["zero_total_anchors"] += 1
        keep = set(info["hard"]) | set(info["soft"]) | {i}
        for j in range(B):
            if j not in keep:
                mask[i, j] = True
    batch_stats["n_batches"] += 1
    batch_stats["n_anchors"] += B

    logits = sim / TEMPERATURE
    neg_inf = torch.finfo(logits.dtype).min
    logits_masked = logits.masked_fill(mask, neg_inf)

    targets = torch.arange(B, device=device)
    loss_t2t = F.cross_entropy(logits_masked, targets)
    loss_t2t_rev = F.cross_entropy(logits_masked.T, targets)
    return (loss_t2t + loss_t2t_rev) / 2

logger.info("Loss function ready.")

2026-10-23 18:43:50,012 : INFO : Loss function ready.


## Section 7 — Load GT set + evaluation function

In [10]:
GT_TEXTTABLE_RECORDS_PATH = os.path.join(VARS_DIR, "gt_texttable_records.pkl")
if not os.path.exists(GT_TEXTTABLE_RECORDS_PATH):
    raise FileNotFoundError(
        f"Missing {GT_TEXTTABLE_RECORDS_PATH}.\n"
        "Run NB 28's Section 6 once to build it, then re-run this cell."
    )

gt_texttable_records = reload_vars(GT_TEXTTABLE_RECORDS_PATH)
logger.info("Loaded %d GT texttable records", len(gt_texttable_records))
assert len(gt_texttable_records) > 0

_s = gt_texttable_records[0]
for _k in ("text", "table_text"):
    assert _k in _s, f"missing key {_k!r} in gt_texttable_records"
logger.info("GT schema OK")

2026-10-23 18:43:50,639 : INFO : Loaded 500 GT texttable records
2026-10-23 18:43:50,641 : INFO : GT schema OK


In [11]:
def compute_retrieval_metrics(sim_matrix, k_values=K_VALUES):
    N = sim_matrix.shape[0]
    true_scores = sim_matrix.diag().unsqueeze(1)
    ranks = (sim_matrix >= true_scores).sum(dim=1)
    results = {}
    for k in k_values:
        results[f"R@{k}"] = (ranks <= k).float().mean().item()
    results["MRR"] = (1.0 / ranks.float()).mean().item()
    results["N"] = N
    return results


def evaluate_on_gt(models):
    models["head_text_bge_v2"].eval()
    models["head_table_bge_v2"].eval()
    models["bge_model"].eval()

    with torch.no_grad():
        gt_text_emb = models["head_text_bge_v2"](
            models["encode_bge_text"]([r["text"] for r in gt_texttable_records])
        )
        gt_table_emb = models["head_table_bge_v2"](
            models["encode_bge_text"]([r["table_text"] for r in gt_texttable_records])
        )

    sim = gt_text_emb @ gt_table_emb.T
    results_text_to_table = compute_retrieval_metrics(sim)
    results_table_to_text = compute_retrieval_metrics(sim.T)

    summary = pd.DataFrame([
        {"direction": "text -> table", **results_text_to_table},
        {"direction": "table -> text", **results_table_to_text},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]

    models["head_text_bge_v2"].train()
    models["head_table_bge_v2"].train()
    models["bge_model"].train()
    return summary

logger.info("evaluate_on_gt() ready.")

2026-10-23 18:43:50,659 : INFO : evaluate_on_gt() ready.


## Section 8 — Sanity check (run before the sweep)

In [12]:
_sanity_models = build_models()
_sanity_batch = texttable_records[:BATCH_SIZE]
_sanity_stats = {"hard_count": 0, "soft_count": 0, "dropped_count": 0, "zero_hard_anchors": 0,
                  "zero_total_anchors": 0, "n_batches": 0, "n_anchors": 0}
loss_check = compute_texttable_loss_inbatch_mined(_sanity_batch, _sanity_models, _sanity_stats)

logger.info("Sanity loss: %.4f (naive ln(%d)=%.4f)", loss_check.item(), BATCH_SIZE, math.log(BATCH_SIZE))
logger.info("Sanity stats: avg hard/anchor=%.1f, avg soft/anchor=%.1f, zero-hard=%d/%d",
            _sanity_stats["hard_count"]/BATCH_SIZE, _sanity_stats["soft_count"]/BATCH_SIZE,
            _sanity_stats["zero_hard_anchors"], BATCH_SIZE)
if not math.isfinite(loss_check.item()):
    raise RuntimeError("Sanity loss is NaN/Inf -- stop and debug")

loss_check.backward()
_grad_bge = any(p.grad is not None and p.grad.abs().sum().item() > 0
                for p in _sanity_models["bge_model"].parameters() if p.requires_grad)
logger.info("Gradients reaching unfrozen BGE layers: %s", _grad_bge)
if not _grad_bge:
    raise RuntimeError("Sanity failed: no gradients reaching unfrozen BGE layers")

del _sanity_models, loss_check, _sanity_batch, _sanity_stats
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
logger.info("Allocated after sanity cleanup: %.2f GB", torch.cuda.memory_allocated(device) / 1e9)

2026-10-23 18:43:55,254 : INFO : build_models: BGE trainable params=96, optimizer groups=2
2026-10-23 18:43:58,808 : INFO : Sanity loss: 3.6102 (naive ln(32)=3.4657)
2026-10-23 18:43:58,811 : INFO : Sanity stats: avg hard/anchor=10.2, avg soft/anchor=19.9, zero-hard=0/32
2026-10-23 18:44:00,871 : INFO : Gradients reaching unfrozen BGE layers: True
2026-10-23 18:44:01,568 : INFO : Allocated after sanity cleanup: 0.02 GB


## Section 9 — Sweep over sizes

In [13]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

sweep_summary_rows = []

for N in SIZES_TO_SWEEP:
    size_dir = os.path.join(OUTPUT_DIR, f"size_{N}")
    os.makedirs(size_dir, exist_ok=True)
    gt_results_path = os.path.join(size_dir, "gt_results.csv")
    ckpt_path = os.path.join(size_dir, "latest_checkpoint.pt")

    if os.path.exists(gt_results_path):
        logger.info("=== N=%d already completed, skipping ===", N)
        sweep_summary_rows.append(pd.read_csv(gt_results_path))
        continue

    logger.info("=" * 70)
    logger.info("=== N=%d ===", N)
    logger.info("=" * 70)

    size_rng = random.Random(RANDOM_SEED + N)
    size_records = list(texttable_records) if N >= FULL_POOL_SIZE else size_rng.sample(texttable_records, N)
    logger.info("Records for this size: %d", len(size_records))

    num_epochs = 20
    models = build_models()

    start_epoch, global_step = 0, 0
    history = {"epoch": [], "texttable_loss": [], "avg_hard_per_anchor": [], "avg_soft_per_anchor": [],
               "zero_hard_anchors": [], "zero_total_anchors": []}

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        models["head_text_bge_v2"].load_state_dict(ckpt["head_text_bge_v2"])
        models["head_table_bge_v2"].load_state_dict(ckpt["head_table_bge_v2"])
        models["bge_model"].load_state_dict(ckpt["bge_model"])
        models["optimizer"].load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"]
        global_step = ckpt["global_step"]
        history = ckpt.get("history", history)
        logger.info("Resumed N=%d from checkpoint: epoch %d, global_step %d", N, start_epoch, global_step)
    else:
        logger.info("No checkpoint found for N=%d, starting fresh.", N)

    def save_checkpoint(epoch, global_step, history):
        tmp_path = ckpt_path + ".tmp"
        torch.save({
            "head_text_bge_v2":  models["head_text_bge_v2"].state_dict(),
            "head_table_bge_v2": models["head_table_bge_v2"].state_dict(),
            "bge_model":         models["bge_model"].state_dict(),
            "optimizer":         models["optimizer"].state_dict(),
            "epoch": epoch, "global_step": global_step, "history": history,
        }, tmp_path)
        os.replace(tmp_path, ckpt_path)

    training_start_time = time.time()
    stop_early = False

    try:
        for epoch in range(start_epoch, num_epochs):
            epoch_losses = []
            epoch_stats = {"hard_count": 0, "soft_count": 0, "dropped_count": 0, "zero_hard_anchors": 0,
                            "zero_total_anchors": 0, "n_batches": 0, "n_anchors": 0}
            batches = list(make_batches(size_records))
            n_steps = len(batches)

            for step in range(n_steps):
                step_start = time.time()
                models["optimizer"].zero_grad()
                loss = compute_texttable_loss_inbatch_mined(batches[step], models, epoch_stats)
                loss.backward()
                models["optimizer"].step()
                global_step += 1
                epoch_losses.append(loss.item())

                elapsed_hours = (time.time() - training_start_time) / 3600
                if step % LOG_EVERY_N_STEPS == 0:
                    step_time = time.time() - step_start
                    logger.info("[N=%d] Epoch %d/%d, step %d/%d, loss=%.4f, step_time=%.2fs, elapsed=%.2fh",
                                N, epoch + 1, num_epochs, step, n_steps, loss.item(), step_time, elapsed_hours)

                if global_step % CHECKPOINT_EVERY_N_STEPS == 0:
                    save_checkpoint(epoch, global_step, history)

                if elapsed_hours > MAX_RUNTIME_HOURS_PER_SIZE:
                    logger.warning("Approaching runtime limit for N=%d. Saving and stopping.", N)
                    save_checkpoint(epoch, global_step, history)
                    stop_early = True
                    break

            if stop_early:
                break

            mean_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
            avg_hard = epoch_stats["hard_count"] / max(epoch_stats["n_anchors"], 1)
            avg_soft = epoch_stats["soft_count"] / max(epoch_stats["n_anchors"], 1)
            history["epoch"].append(epoch + 1)
            history["texttable_loss"].append(mean_loss)
            history["avg_hard_per_anchor"].append(avg_hard)
            history["avg_soft_per_anchor"].append(avg_soft)
            history["zero_hard_anchors"].append(epoch_stats["zero_hard_anchors"])
            history["zero_total_anchors"].append(epoch_stats["zero_total_anchors"])
            logger.info("[N=%d] Epoch %d done: loss=%.4f, avg_hard/anchor=%.1f, avg_soft/anchor=%.1f, "
                        "zero_hard=%d, zero_total=%d",
                        N, epoch + 1, mean_loss, avg_hard, avg_soft,
                        epoch_stats["zero_hard_anchors"], epoch_stats["zero_total_anchors"])

            gt_summary = evaluate_on_gt(models)
            gt_summary.insert(0, "train_size", N)
            logger.info("GT results for N=%d:\n%s", N, gt_summary.to_string(index=False))
            save_checkpoint(epoch + 1, global_step, history)

        pd.DataFrame(history).to_csv(os.path.join(size_dir, "training_history.csv"), index=False)
        for key in ("head_text_bge_v2", "head_table_bge_v2", "bge_model"):
            fname_stem = _MODEL_FILE_RENAMES.get(key, key)
            torch.save(models[key].state_dict(), os.path.join(size_dir, f"{fname_stem}.pt"))
        logger.info("Saved outputs for N=%d to %s", N, size_dir)

        gt_summary = evaluate_on_gt(models)
        gt_summary.insert(0, "train_size", N)
        gt_summary.to_csv(gt_results_path, index=False)
        logger.info("GT results for N=%d:\n%s", N, gt_summary.to_string(index=False))
        sweep_summary_rows.append(gt_summary)

    except torch.cuda.OutOfMemoryError:
        logger.error("OOM on N=%d. Skipping to next size.", N)

    finally:
        try:
            del models
        except NameError:
            pass
        gc.collect()
        if torch.cuda.is_available():
            with torch.cuda.device(CUDA):
                torch.cuda.empty_cache()
                torch.cuda.synchronize()

logger.info("=" * 70)
logger.info("Sweep complete.")
logger.info("=" * 70)

2026-10-23 18:44:01,586 : INFO : === N=100 already completed, skipping ===
2026-10-23 18:44:01,591 : INFO : === N=500 already completed, skipping ===
2026-10-23 18:44:01,596 : INFO : === N=1000 already completed, skipping ===
2026-10-23 18:44:01,600 : INFO : === N=2500 already completed, skipping ===
2026-10-23 18:44:01,604 : INFO : === N=5000 already completed, skipping ===
2026-10-23 18:44:01,608 : INFO : ======================================================================
2026-10-23 18:44:01,610 : INFO : === N=6374 ===
2026-10-23 18:44:01,611 : INFO : ======================================================================
2026-10-23 18:44:01,612 : INFO : Records for this size: 6374
2026-10-23 18:44:04,130 : INFO : build_models: BGE trainable params=96, optimizer groups=2
2026-10-23 18:44:11,897 : INFO : Resumed N=6374 from checkpoint: epoch 19, global_step 3800
2026-10-23 18:44:16,736 : INFO : [N=6374] Epoch 20/20, step 0/200, loss=0.0328, step_time=4.83s, elapsed=0.00h
2026-10-23 

## Section 10 — Combined summary across sizes

In [14]:
rows = []
for N in SIZES_TO_SWEEP:
    path = os.path.join(OUTPUT_DIR, f"size_{N}", "gt_results.csv")
    if os.path.exists(path):
        rows.append(pd.read_csv(path))
    else:
        logger.warning("N=%d: no gt_results.csv found, skipping", N)

if rows:
    summary_all_sizes = (pd.concat(rows, ignore_index=True)
                           .sort_values(["train_size", "direction"])
                           .reset_index(drop=True))
    summary_all_sizes.to_csv(os.path.join(OUTPUT_DIR, "summary_all_sizes.csv"), index=False)
    logger.info("=== Full summary across sizes ===\n%s", summary_all_sizes.to_string(index=False))
    try:
        display(summary_all_sizes)
    except NameError:
        print(summary_all_sizes)
else:
    logger.warning("No gt_results.csv files found; nothing to summarize.")

2026-10-23 19:03:15,457 : INFO : === Full summary across sizes ===
 train_size     direction   N   R@1   R@3   R@5  R@10      MRR
        100 table -> text 500 0.062 0.134 0.180 0.274 0.131083
        100 text -> table 500 0.048 0.142 0.204 0.292 0.130226
        500 table -> text 500 0.140 0.328 0.432 0.554 0.274146
        500 text -> table 500 0.152 0.310 0.402 0.516 0.271955
       1000 table -> text 500 0.206 0.406 0.506 0.616 0.340675
       1000 text -> table 500 0.188 0.428 0.504 0.616 0.340034
       2500 table -> text 500 0.298 0.524 0.646 0.752 0.445119
       2500 text -> table 500 0.276 0.514 0.620 0.742 0.429200
       5000 table -> text 500 0.358 0.602 0.714 0.826 0.510323
       5000 text -> table 500 0.362 0.606 0.714 0.812 0.518552
       6374 table -> text 500 0.384 0.664 0.746 0.836 0.545056
       6374 text -> table 500 0.376 0.642 0.750 0.836 0.539193


,train_size,direction,N,R@1,R@3,R@5,R@10,MRR
0,100,table -> text,500,0.062,0.134,0.180,0.274,0.131083
1,100,text -> table,500,0.048,0.142,0.204,0.292,0.130226
2,500,table -> text,500,0.140,0.328,0.432,0.554,0.274146
3,500,text -> table,500,0.152,0.310,0.402,0.516,0.271955
4,1000,table -> text,500,0.206,0.406,0.506,0.616,0.340675
5,1000,text -> table,500,0.188,0.428,0.504,0.616,0.340034
6,2500,table -> text,500,0.298,0.524,0.646,0.752,0.445119
7,2500,text -> table,500,0.276,0.514,0.620,0.742,0.429200
8,5000,table -> text,500,0.358,0.602,0.714,0.826,0.510323
9,5000,text -> table,500,0.362,0.606,0.714,0.812,0.518552
